# Study in Germany Assistant: run in Colab
Runtime → Change runtime type → **T4 GPU**. This notebook only calls the code in `Assignment 1/Code/`.

Everything is kept on Google Drive (`MyDrive/nlp-tasks-2026/Assignment 1/`):
code, data, indexes, refusal calibration files and results.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Option A (active): run from Google Drive, so every file created is kept
PROJECT = "/content/drive/MyDrive/nlp-tasks-2026/Assignment 1"
%cd "$PROJECT/Code"

# Option B: fresh clone from GitHub (files created are lost when the session ends)
# !git clone -q https://github.com/cedricfyc/nlp-tasks-2026.git /content/nlp-tasks-2026
# PROJECT = "/content/nlp-tasks-2026/Assignment 1"
# %cd "$PROJECT/Code"

# docling is only needed to rebuild the database from the PDFs and is slow to install: skipped here
!grep -v docling ../requirements.txt > /tmp/requirements_run.txt
!pip install -q -r /tmp/requirements_run.txt

Mounted at /content/drive
/content/drive/MyDrive/nlp-tasks-2026/Assignment 1/Code


### Where to get Qwen3.5-4B (about 8 GB)
- **Option A (active):** the model is already on Google Drive and is loaded from there.
- **Option B (commented):** download from Hugging Face. To use it, comment out Option A and
  uncomment the three Option B lines. `snapshot_download` skips files that are already there.

In [3]:
import os

# Option A: Google Drive (model already stored on Drive)
os.environ["RAG_MODEL_PATH"] = "/content/drive/MyDrive/models/Qwen3.5-4B"

# Option B: Hugging Face (no Drive space needed; downloads again every session, ~3-10 min)
# from huggingface_hub import snapshot_download
# os.environ["RAG_MODEL_PATH"] = "/content/models/Qwen3.5-4B"
# snapshot_download("Qwen/Qwen3.5-4B", local_dir=os.environ["RAG_MODEL_PATH"])

# Keep evaluation results on Drive. The Hugging Face models (embedding, SPLADE, re-ranker) are
# downloaded to the Colab disk each session (~1-2 min): caching them on Drive corrupts large files.
os.environ["RAG_RESULTS_DIR"] = f"{PROJECT}/results"

# Hugging Face login: needed for SPLADE (naver/splade-v3 is gated: accept the licence on its page first).
# Asked once per session.
from huggingface_hub import get_token, login
if get_token() is None:
    login()

## 2. One-time steps
Each step saves its file in `data/` on Drive and **skips itself** when the file already exists,
so this cell can always be run (add `--force` to redo one).

In [4]:
!python build_database.py --splade                 # data/splade_index.npz (+ chunk ids)
!python calibrate_refusal.py --method bm25         # data/refusal_bm25.json (already in the repo, skips)
!python calibrate_refusal.py --method dense        # data/refusal_dense.json
!python calibrate_refusal.py --method llm_judge --retriever both --splade --reranker   # data/refusal_llm_judge.json

SPLADE index already exists (/content/drive/MyDrive/nlp-tasks-2026/Assignment 1/data/splade_index.npz); skipped. Use --force to rebuild.
/content/drive/MyDrive/nlp-tasks-2026/Assignment 1/data/refusal_bm25.json already exists; skipped. Use --force to calibrate again.
/content/drive/MyDrive/nlp-tasks-2026/Assignment 1/data/refusal_dense.json already exists; skipped. Use --force to calibrate again.
/content/drive/MyDrive/nlp-tasks-2026/Assignment 1/data/refusal_llm_judge.json already exists; skipped. Use --force to calibrate again.


## 3. Evaluation (answers + metrics + LLM judge in `results/` on Drive)
Run **before** section 4: each command loads its own copy of the model, and two copies do not fit on a T4.

Three runs on the **60 test questions** (`data/test_60.json`, not used for calibration; about 40-70 min each on a T4):
1. **BM25 baseline**
2. **All retrievers, no refusal check** (BM25 + dense + SPLADE + re-ranker, hard citations)
3. **Full system end to end** (2. + combined refusal: majority vote of bm25, dense and llm_judge)

1 vs 2 shows what the retrieval adds; 2 vs 3 shows what the refusal check adds.
`--judge` only grades the answers (it is not part of the system).
Every run is **resumable**: if Colab disconnects, run the cell again.

In [4]:
TEST = "--dataset ../data/test_60.json"     # 60 questions not used for calibration

!python run_eval.py --retriever bm25 $TEST                                           # 1. baseline
!python run_eval.py --retriever both --splade --reranker $TEST                     # 2. all retrievers, no refusal check
!python run_eval.py --retriever both --splade --reranker --refusal combined $TEST    # 3. full system end to end

# Single refusal methods:
# !python run_eval.py --retriever both --splade --reranker --refusal bm25 $TEST --judge
# !python run_eval.py --retriever both --splade --reranker --refusal dense $TEST --judge
# !python run_eval.py --retriever both --splade --reranker --refusal llm_judge $TEST --judge
# Retrieval one change at a time:
# !python run_eval.py --retriever dense $TEST --judge                   # dense only
# !python run_eval.py --retriever both $TEST --judge                    # BM25 + dense
# !python run_eval.py --retriever both --splade $TEST --judge           # + SPLADE
# !python run_eval.py --retriever both --reranker $TEST --judge         # + re-ranker without SPLADE
# Same runs on the 100 calibration questions: leave out $TEST

Setting: bm25_refusal-none_test_60
Results: /content/drive/MyDrive/nlp-tasks-2026/Assignment 1/results
GPU: Tesla T4
Loading weights: 100% 426/426 [00:34<00:00, 12.38it/s]
Loaded BM25 index with 3,719 chunks (built 2026-11-18T20:28:29.422367+00:00).
Loading weights: 100% 310/310 [00:00<00:00, 480.07it/s]
60 questions to answer (0 already saved in /content/drive/MyDrive/nlp-tasks-2026/Assignment 1/results/rag_bm25_refusal-none_test_60.db)

Generating...

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implement

## 4. Load the pipeline and start the UI + `POST /ask`

In [6]:
from pipeline import RAGPipeline
from api import start_in_background

rag = RAGPipeline(use_splade=True, use_reranker=True,   # True after the evaluation shows a gain
                  refusal_method="combined",                  # "bm25" | "dense" | "llm_judge" | "combined"
                  retriever="both")                       # "bm25" | "dense"
start_in_background(rag)

SystemExit: No CUDA GPU found.

## 5. Ask from the notebook

In [6]:
thinking, answer = rag.conduct_rag("What are the requirements for a blocked account in Germany?")
rag.last_context_df   # shows rerank_score as well when the re-ranker is on

,chunk_id,doc_id,filename,source_title,section,page_start,page_end,text,dense_score,dense_rank,bm25_score,bm25_rank,in_both
0,1494,41,Studying in Germany - A Comprehensive Guide,Expatrio,What is a Blocked Account?,16.0,17.0,A Blocked Account (Sperrkonto) is a special ac...,0.805738,1.0,29.300601,1.0,True
1,1493,41,Studying in Germany - A Comprehensive Guide,Expatrio,3 Open a Blocked Account,16.0,16.0,An important requirement for the issuance of a...,0.705166,3.0,26.193507,2.0,True
2,1529,42,How to Study in Germany,Study in Germany,ssecorP﻿ 3.2. Visa Application,43.0,43.0,"""7\nZOE\nL 77\nZOE\nExactly. You'll need to sh...",0.724055,2.0,NaN,NaN,False
3,1530,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.758687,3.0,False


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


Top-3 overlap: 2 chunks

Generating...



[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


To qualify for a visa or residence permit in Germany, applicants must demonstrate that their livelihood is secured for the duration of their stay [2]. The primary groups required to have a Blocked Account are students, language students, job seekers, and anyone whose stay is not supported by external sources like a salary or scholarship [1]. The account must contain a minimum amount set by law, which was €992 per month in 2024 [1]. This total amount must be deposited in full when the account is opened, for example, €11,208 for a 12-month stay [3]. The funds remain blocked until the holder arrives in Germany, at which point the account is activated [1]. Once activated, the holder receives monthly payouts transferred to a separate regular bank account opened in Germany [4].<|im_end|>
<|endoftext|>

Generation complete.

--- Cited answer ---
To qualify for a visa or residence permit in Germany, applicants must demonstrate that their livelihood is secured for the duration of their stay [1]

,chunk_id,doc_id,chunk_index,section,page_start,page_end,text,filename,source_url,download_url,source_title,source_type
0,1494,41,37,What is a Blocked Account?,16,17,A Blocked Account (Sperrkonto) is a special ac...,Studying in Germany - A Comprehensive Guide,https://www.expatrio.com/,https://www.expatrio.com/library/study-in-germany,Expatrio,PDF
1,1493,41,36,3 Open a Blocked Account,16,16,An important requirement for the issuance of a...,Studying in Germany - A Comprehensive Guide,https://www.expatrio.com/,https://www.expatrio.com/library/study-in-germany,Expatrio,PDF
2,1529,42,30,ssecorP﻿ 3.2. Visa Application,43,43,"""7\nZOE\nL 77\nZOE\nExactly. You'll need to sh...",How to Study in Germany,https://www.studying-in-germany.org/,https://lp.studying-in-germany.org/ebook/,Study in Germany,PDF
3,1530,42,31,LL 77,43,43,MIA\nLL 77\nHere's how it works: The German Bl...,How to Study in Germany,https://www.studying-in-germany.org/,https://lp.studying-in-germany.org/ebook/,Study in Germany,PDF


## 6. Other tasks

In [7]:
!python compare_retrievers.py                    # BM25 vs dense side by side
# !python build_database.py --bm25                # rebuild BM25 after the database changes
# !python run_eval.py --retriever bm25 --metrics-only   # recompute metrics from saved answers

Loaded BM25 index with 3,719 chunks (built 2026-11-18T20:28:29.422367+00:00).
Loaded vectors: (3719, 1024)
Loading weights: 100% 310/310 [00:00<00:00, 19446.24it/s]
    chunk_id doc_id  ... bm25_rank in_both
0        255     14  ...       2.0    True
1        195     10  ...       1.0    True
2         96      7  ...       NaN   False
3        818     36  ...       NaN   False
4        758     36  ...       NaN   False
5       1128     39  ...       NaN   False
6        759     36  ...       NaN   False
7       2972     74  ...       NaN   False
8       1176     39  ...       NaN   False
9        805     36  ...       NaN   False
10      2901    NaN  ...       3.0   False
11      1072    NaN  ...       4.0   False
12      1073    NaN  ...       5.0   False
13      1075    NaN  ...       6.0   False
14      1109    NaN  ...       7.0   False
15       194    NaN  ...       8.0   False
16       192    NaN  ...       9.0   False
17       199    NaN  ...      10.0   False

[18 rows x 13 col